In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from data_tools import process_data as processed
from data_tools.combine_data import CreateTrainingandTestData
from data_tools import process_dates
from data_tools import handle_datetime
from data_tools import clean_data as clean

combine = CreateTrainingandTestData()
# Change as needed
sensor_id = "Coalinga"
datatype = "o3"
year = "2025"

training_dates = process_dates.training(sensor_id, year)
testing_dates= process_dates.testing(sensor_id, year)

# Define files that hold necessary raw data
reference_file_1 = rf"../reference_files/OZONE_PICKDATA_2025-7-29-Fresno.csv"
reference_file_2 = rf"../reference_files/AQLite-2025-11-06.csv"
sensor_file = rf"../reference_files/2025rawdata/{sensor_id}.csv"
precal_reference = handle_datetime.utc_to_CA(processed.ref_data(reference_file_1))
postcal_reference = processed.standard_data(reference_file_2)
postcal_reference = postcal_reference.resample('h').mean()
postcal_reference.index = postcal_reference.index.map(lambda ts: ts.replace(minute=30, second=0, microsecond=0))
postcal_reference = handle_datetime.utc_to_CA(postcal_reference)
voz_data = handle_datetime.utc_to_CA(processed.raw_voz_data(sensor_file))

when = {
    "precal_start": training_dates[0],
    "precal_end": training_dates[1],
    "postcal_start": training_dates[2],
    "postcal_end": training_dates[3],
    "trial1_start": testing_dates[0],
    "trial1_end": testing_dates[1],
    "trial2_start": testing_dates[2],
    "trial2_end": testing_dates[3]
}

combine.set_calibration_parameters(datatype,sensor_id,voz_data)
training_data,all_data = combine.get_combined_data(training_dates,testing_dates,precal_reference,postcal_reference)
training_data, all_data = map(lambda df: clean.eliminate_waste_data_o3(df), [training_data, all_data])

                     Unnamed: 0.1  Unnamed: 0  m_PM1_CF1  m_PM1_ATM  m_PM1_b  \
date_time                                                                      
2025-06-27 12:30:00             3         NaN        NaN        NaN      NaN   
2025-06-27 13:30:00             4         NaN        NaN        NaN      NaN   
2025-06-27 14:30:00             5         NaN        NaN        NaN      NaN   
2025-06-27 15:30:00             6         NaN        NaN        NaN      NaN   
2025-06-27 16:30:00             7         NaN        NaN        NaN      NaN   
...                           ...         ...        ...        ...      ...   
2025-11-04 15:30:00          2829         NaN        NaN        NaN      NaN   
2025-11-04 16:30:00          2830         NaN        NaN        NaN      NaN   
2025-11-04 17:30:00          2831         NaN        NaN        NaN      NaN   
2025-11-04 18:30:00          2832         NaN        NaN        NaN      NaN   
2025-11-04 19:30:00          2833       